# Business Forecasting | ARIMA Forecasting in Python

**Student Colab notebook**  
This lesson follows Chapter 9 of *Forecasting: Principles and Practice, the Pythonic Way* (FPPpy). We will use the book's annual Central African Republic exports series to connect differencing, ACF/PACF, ARIMA model choice, residual checks, and forecasts. A short optional section introduces seasonal ARIMA.

**Learning goals**

By the end, you should be able to explain the three ARIMA orders, decide whether differencing may be needed, use ACF and PACF to suggest candidate models, compare candidate fits with AICc, check whether residuals retain autocorrelation, and explain what forecast intervals mean.

**How to use the notebook:** In Colab, choose **Runtime → Run all**. The notebook downloads the book datasets, so an internet connection is needed. Code outputs may vary slightly across package versions.

## 1. Set up the notebook

The chapter uses StatsForecast for ARIMA estimation and Statsmodels for ACF/PACF plots and the Ljung–Box test. The installation cell is needed only in a fresh Colab runtime.

In [ ]:
%pip -q install statsforecast statsmodels

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.precision", 3)
np.random.seed(477)

## 2. Load the book data

The source chapter uses the `global_economy.csv` file and selects the `Exports` series for the Central African Republic (`CAF`). Exports are measured as a **percentage of GDP**, not as a dollar amount. The file is hosted by OTexts.

In [ ]:
DATA_URL = "https://otexts.com/fpppy/data/global_economy.csv"

try:
    economy = pd.read_csv(DATA_URL, parse_dates=[2])
except Exception as exc:
    raise RuntimeError(
        "Could not download the OTexts data. Check your internet connection "
        "and try again. Source: " + DATA_URL
    ) from exc

caf = (
    economy.rename(columns={"Exports": "y"})
    .loc[lambda df: df["Code"] == "CAF", ["unique_id", "ds", "y"]]
    .dropna()
    .sort_values("ds")
    .reset_index(drop=True)
)

if caf.empty:
    raise ValueError("No CAF rows were found. Check the downloaded file's columns and contents.")

print(f"Rows: {len(caf)} | Years: {caf['ds'].min().year}–{caf['ds'].max().year}")
caf.head()

## 3. Look at the series before fitting a model

A time plot can reveal trend, seasonality, changes in spread, and unusual observations. The source chapter describes a general decline in exports as a share of GDP, so a first difference is a reasonable starting point. A plot alone does not prove stationarity.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(caf["ds"], caf["y"], color="#176B87", marker="o", markersize=3)
ax.set(title="Central African Republic exports", xlabel="Year", ylabel="Exports (% of GDP)")
plt.show()

## 4. Differencing and stationarity

A first difference is the current value minus the previous value: $y_t-y_{t-1}$. It often removes a changing level or trend. Differencing too many times can add noise, so use the smallest order that makes the series reasonably stable.

In [ ]:
caf_diff = caf.assign(diff=caf["y"].diff()).dropna()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(caf["ds"], caf["y"], color="#176B87")
axes[0].set(title="Original series", ylabel="Exports (% of GDP)")
axes[1].plot(caf_diff["ds"], caf_diff["diff"], color="#D2691E")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="First difference", xlabel="Year", ylabel="Change")
plt.tight_layout()
plt.show()

**Interpretation check:** Does the differenced series fluctuate around a fairly stable level? Are there large changes that deserve investigation? Differencing changes the question: the model now describes annual changes in the export share, and forecasts are converted back to the original scale.

## 5. Use ACF and PACF to suggest model orders

The ACF measures correlation between observations separated by a lag. The PACF measures the additional correlation at a lag after accounting for shorter lags. For the differenced series, a PACF that cuts off after lag $p$ can suggest AR($p$); an ACF that cuts off after lag $q$ can suggest MA($q$). These are clues, not automatic rules, especially when both AR and MA terms may be present.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(caf_diff["diff"], lags=15, zero=False, ax=axes[0])
plot_pacf(caf_diff["diff"], lags=15, zero=False, method="ywm", ax=axes[1])
axes[0].set_title("ACF of first differences")
axes[1].set_title("PACF of first differences")
plt.tight_layout()
plt.show()

**Candidate models from the source chapter:** the differenced-series PACF suggests ARIMA(2,1,0), while the ACF suggests ARIMA(0,1,3). We will compare these with automatic stepwise and broader searches. Do not select a model from a single plot alone.

## 6. Fit and compare candidate ARIMA models

The order is written $(p,d,q)$: $p$ is the number of autoregressive lags, $d$ is the number of ordinary differences, and $q$ is the number of lagged forecast errors. ARIMA(2,1,0) and ARIMA(0,1,3) are theory-guided candidates here. `AutoARIMA` searches for other combinations.

In [ ]:
models = [
    ARIMA(order=(2, 1, 0), alias="ARIMA(2,1,0)"),
    ARIMA(order=(0, 1, 3), alias="ARIMA(0,1,3)"),
    AutoARIMA(alias="Stepwise AutoARIMA"),
    AutoARIMA(stepwise=False, alias="Broader AutoARIMA search"),
]

sf = StatsForecast(models=models, freq="A", n_jobs=-1)
sf.fit(df=caf)

rows = []
for fitted_model in sf.fitted_[0]:
    result = fitted_model.model_
    rows.append({
        "model": fitted_model.alias,
        "AIC": result["aic"],
        "AICc": result["aicc"],
        "BIC": result["bic"],
    })

comparison = pd.DataFrame(rows).sort_values("AICc").reset_index(drop=True)
comparison.round(2)

### Reading the comparison

AICc rewards fit while penalizing extra parameters, with an additional correction for smaller samples. Among models fitted to the same response data with the same differencing, a lower AICc is preferred as a fit-versus-complexity guide. It does **not** prove that the model will forecast best. AICc values should not be used to choose $d$ across models fitted to differently differenced data.

## 7. Check the chosen model

A useful ARIMA model leaves residuals that look like white noise: no visible pattern over time, no important remaining autocorrelation, and a distribution without severe problems. We use the broader-search model below to follow the chapter's example. If your version selects a different order, record that order and use its $p+q$ count in the Ljung–Box adjustment.

In [ ]:
forecast_table = sf.forecast(df=caf, h=10, fitted=True, level=[80, 95])
in_sample = sf.forecast_fitted_values()

search_name = "Broader AutoARIMA search"
residuals = (in_sample["y"] - in_sample[search_name]).dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].plot(in_sample["ds"], residuals, color="#176B87")
axes[0, 0].axhline(0, color="black", linewidth=0.8)
axes[0, 0].set(title="Residuals over time", xlabel="Year", ylabel="Residual")
plot_acf(residuals, lags=12, zero=False, ax=axes[0, 1])
axes[0, 1].set_title("Residual ACF")
axes[1, 0].hist(residuals, bins=12, color="#D2691E", edgecolor="white")
axes[1, 0].set(title="Residual distribution", xlabel="Residual", ylabel="Count")
axes[1, 1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# The source chapter uses model_df=3 for its selected ARIMA(3,1,0).
# If your automatically selected p and q differ, update model_df to p + q.
ljung_box = acorr_ljungbox(residuals, lags=[10], model_df=3, return_df=True)
ljung_box

**Decision:** A large Ljung–Box p-value means we do not have strong evidence of remaining autocorrelation at the tested lags. It does not prove that all assumptions hold. Inspect the residual plot and ACF as well. If residual structure remains, revise the model before relying on its forecasts.

## 8. Forecast and communicate uncertainty

Prediction intervals show a range of plausible future values under the model assumptions. Wider intervals at longer horizons are common, especially after differencing. The interval does not include every source of uncertainty, such as future structural changes or uncertainty about which model is best.

In [ ]:
forecast_table.head()

In [ ]:
# The forecast columns are produced by StatsForecast for each model.
# Inspect the names once in case a future package version changes its naming convention.
print(forecast_table.columns.tolist())

name = search_name
point = forecast_table[name]
lower_candidates = [c for c in forecast_table.columns if c.startswith(name) and "lo-80" in c]
upper_candidates = [c for c in forecast_table.columns if c.startswith(name) and "hi-80" in c]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(caf["ds"], caf["y"], color="black", label="Observed")
ax.plot(forecast_table["ds"], point, color="#176B87", marker="o", label="Forecast")
if lower_candidates and upper_candidates:
    ax.fill_between(
        forecast_table["ds"],
        forecast_table[lower_candidates[0]],
        forecast_table[upper_candidates[0]],
        color="#8CC6D7", alpha=0.35, label="80% prediction interval"
    )
ax.set(title="Ten-year ARIMA forecast", xlabel="Year", ylabel="Exports (% of GDP)")
ax.legend()
plt.show()

## 9. Optional extension: seasonal ARIMA

For monthly or quarterly data, seasonal ARIMA adds seasonal orders $(P,D,Q)_m$ to $(p,d,q)$. The seasonal period $m$ is 12 for monthly data with annual seasonality and 4 for quarterly data. The source chapter uses monthly U.S. leisure and hospitality employment and reports that the broader search selected ARIMA(2,1,0)(0,1,2)$_{12}$. This section is optional if seasonal ARIMA is outside the current course scope.

In [ ]:
EMPLOYMENT_URL = "https://otexts.com/fpppy/data/us_employment.csv"

try:
    employment = pd.read_csv(EMPLOYMENT_URL, parse_dates=["ds"])
except Exception as exc:
    raise RuntimeError(
        "Could not download the OTexts employment data. Check your internet connection."
    ) from exc

leisure = (
    employment.loc[
        (employment["unique_id"] == "Leisure and Hospitality")
        & (employment["ds"] >= "2001"),
        ["unique_id", "ds", "y"]
    ]
    .sort_values("ds")
    .reset_index(drop=True)
)
leisure["y"] = leisure["y"] / 1_000  # convert thousands of people to millions

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(leisure["ds"], leisure["y"], color="#176B87")
ax.set(title="U.S. leisure and hospitality employment", xlabel="Month", ylabel="People (millions)")
plt.show()

In [ ]:
seasonal_model = AutoARIMA(stepwise=False, approximation=False, season_length=12)
seasonal_sf = StatsForecast(models=[seasonal_model], freq="MS", n_jobs=-1)
seasonal_sf.fit(df=leisure)
seasonal_forecast = seasonal_sf.forecast(df=leisure, h=24, level=[80, 95])
seasonal_forecast.head()

## 10. ARIMA and ETS

ARIMA and exponential smoothing (ETS) are complementary forecasting families. ARIMA describes autocorrelation after differencing; ETS describes level, trend, and seasonality components. AICc is appropriate for comparing ARIMA candidates or ETS candidates separately, but it should not be used to compare likelihoods across the two families. For a business decision, compare genuine out-of-sample forecasts using a time-based holdout or rolling-origin validation.

## 12. Worked example: simulated monthly sales

The next section matches the step-by-step lecture example. It creates a reproducible **simulated** sales series, keeps the final 12 months as a test set, fits ARIMA(1,1,0) to the differenced training data with conditional least squares, checks residuals, and forecasts the holdout. These values are for learning only and do not describe a real company.

In [ ]:
rng = np.random.default_rng(477)
n = 120
true_drift, true_phi, innovation_sd = 1.1, 0.55, 2.8
innovations = rng.normal(0, innovation_sd, n)
monthly_change = np.zeros(n)
monthly_change[0] = true_drift + innovations[0]

for t in range(1, n):
    monthly_change[t] = (
        true_drift
        + true_phi * (monthly_change[t - 1] - true_drift)
        + innovations[t]
    )

sales = 100 + np.cumsum(monthly_change)
dates = pd.date_range("2015-01-01", periods=n, freq="MS")
sales_df = pd.DataFrame({"ds": dates, "sales": sales})

### Step 1: plot before modelling

Look for a changing level, unusual observations, gaps, and changing spread. Here the sales level rises because monthly changes average above zero.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sales_df["ds"], sales_df["sales"], color="#176B87")
ax.set(title="Simulated monthly sales", xlabel="Month", ylabel="Sales index")
plt.show()

### Step 2: reserve a test period

We keep the final 12 months out of model estimation. The test data will tell us how the selected model forecasts observations it did not see.

In [ ]:
h = 12
cut = n - h
train_sales = sales[:cut]
test_sales = sales[cut:]
train_dates = dates[:cut]
test_dates = dates[cut:]

print(f"Training observations: {len(train_sales)}")
print(f"Test observations: {len(test_sales)}")

### Step 3: difference the training values

A first difference converts the series into month-to-month changes. The training series has one fewer difference than observations.

In [ ]:
train_change = np.diff(train_sales)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(train_dates, train_sales, color="#176B87")
axes[0].set(title="Training sales level", ylabel="Sales index")
axes[1].plot(train_dates[1:], train_change, color="#D2691E")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="First differences", xlabel="Month", ylabel="Monthly change")
plt.tight_layout()
plt.show()

### Step 4: use ACF and PACF to suggest a model

The PACF can suggest a short AR component when it drops off after the early lags. These plots are clues. We compare several candidates rather than selecting a model from one plot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_acf(train_change, lags=12, zero=False, ax=axes[0])
plot_pacf(train_change, lags=12, zero=False, method="ywm", ax=axes[1])
axes[0].set_title("ACF of training changes")
axes[1].set_title("PACF of training changes")
plt.tight_layout()
plt.show()

### Step 5: compare candidate models with AICc

The code below fits candidate models to the same training series. A lower AICc is a useful fit-versus-complexity guide among these candidates. We still need to inspect residuals and test-set accuracy.

In [ ]:
sales_long = pd.DataFrame({
    "unique_id": "monthly_sales",
    "ds": train_dates,
    "y": train_sales,
})

candidate_models = [
    ARIMA(order=(0, 1, 0), alias="ARIMA(0,1,0)"),
    ARIMA(order=(1, 1, 0), alias="ARIMA(1,1,0)"),
    ARIMA(order=(0, 1, 1), alias="ARIMA(0,1,1)"),
    ARIMA(order=(1, 1, 1), alias="ARIMA(1,1,1)"),
]
sales_sf = StatsForecast(models=candidate_models, freq="MS", n_jobs=-1)
sales_sf.fit(df=sales_long)

model_names = ["ARIMA(0,1,0)", "ARIMA(1,1,0)", "ARIMA(0,1,1)", "ARIMA(1,1,1)"]
comparison_rows = []
for fitted_model, name in zip(sales_sf.fitted_[0], model_names):
    fit = fitted_model.model_
    comparison_rows.append({"model": name, "AICc": fit["aicc"], "AIC": fit["aic"]})

comparison = pd.DataFrame(comparison_rows).sort_values("AICc")
comparison.round(2)

### Step 6: estimate ARIMA(1,1,0) and inspect residuals

This short conditional least-squares calculation makes the AR(1) structure in the differences visible. It may return slightly different coefficients than StatsForecast, which estimates models using likelihood methods.

In [ ]:
X = np.column_stack([np.ones(len(train_change) - 1), train_change[:-1]])
beta = np.linalg.lstsq(X, train_change[1:], rcond=None)[0]
c_hat, phi_hat = beta
residuals = train_change[1:] - X @ beta
sigma_hat = np.sqrt(np.mean(residuals ** 2))
long_run_change = c_hat / (1 - phi_hat)

print(f"Estimated long-run monthly change: {long_run_change:.2f}")
print(f"Estimated AR coefficient: {phi_hat:.2f}")
print(f"Residual standard deviation: {sigma_hat:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_dates[2:], residuals, color="#176B87")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Residuals over time", xlabel="Month", ylabel="Residual")
plot_acf(residuals, lags=12, zero=False, ax=axes[1])
axes[1].set_title("Residual ACF")
plt.tight_layout()
plt.show()

acorr_ljungbox(residuals, lags=[10], model_df=1, return_df=True)

### Step 7: forecast and evaluate the 12-month test set

The ARIMA(1,1,0) forecast for the change gradually returns toward its long-run mean. We accumulate the forecasted changes to obtain sales levels. The intervals use the model’s innovation variance and widen as the horizon grows.

In [ ]:
horizon = np.arange(1, h + 1)
change_forecast = long_run_change + phi_hat ** horizon * (train_change[-1] - long_run_change)
sales_forecast = train_sales[-1] + np.cumsum(change_forecast)

forecast_variance = [
    sigma_hat ** 2 * sum(
        ((1 - phi_hat ** r) / (1 - phi_hat)) ** 2
        for r in range(1, j + 1)
    )
    for j in horizon
]
forecast_se = np.sqrt(forecast_variance)
lower80 = sales_forecast - 1.2816 * forecast_se
upper80 = sales_forecast + 1.2816 * forecast_se

mae = np.mean(np.abs(test_sales - sales_forecast))
rmse = np.sqrt(np.mean((test_sales - sales_forecast) ** 2))
print(f"12-month holdout MAE: {mae:.2f}")
print(f"12-month holdout RMSE: {rmse:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(train_dates, train_sales, color="black", label="Training sales")
ax.plot(test_dates, test_sales, color="#D2691E", marker="o", label="Held-out actual")
ax.plot(test_dates, sales_forecast, color="#176B87", marker="o", label="ARIMA(1,1,0) forecast")
ax.fill_between(test_dates, lower80, upper80, color="#8CC6D7", alpha=0.35, label="80% interval")
ax.set(title="12-month forecast and holdout", xlabel="Month", ylabel="Sales index")
ax.legend()
plt.show()

**Interpretation:** The holdout errors apply only to this simulated series and chosen 12-month split. They do not estimate the forecast accuracy of a real business. For a real project, use data with meaningful units, repeat the evaluation across rolling forecast origins, and compare against a simple baseline.

## 11. Practice and reflection

1. In the main example, what does $d=1$ mean in business terms?  
2. Why do the ACF and PACF suggest candidates rather than identify a final model?  
3. Compare the four candidates using AICc. What does the winning value tell you, and what does it not tell you?  
4. Interpret the Ljung–Box result using the 5% significance level.  
5. How would you explain an 80% prediction interval to a manager?  
6. Name one limitation of forecasting an export share from historical annual data.  
7. For the optional seasonal series, what does $m=12$ represent?

## Sources and data limitations

- Hyndman, R. J., Athanasopoulos, G., Garza, M., Challu, C., & coauthors. *Forecasting: Principles and Practice, the Pythonic Way*, Chapter 9, ‘ARIMA models.’ https://otexts.com/fpppy/09-arima.html  
- Book data: `global_economy.csv` and `us_employment.csv`, hosted at https://otexts.com/fpppy/data/  
- Data coverage follows the book's published examples and is not a live or regularly updated business feed. The export series is an annual ratio (% of GDP), which can change because either exports or GDP changes. Historical relationships may not continue. The notebook needs internet access to retrieve the CSV files.  
- This is a teaching workflow, not a forecast for policy or investment decisions. Use current, domain-appropriate data and out-of-sample evaluation for an applied project.